# **Demo 02: Executing Serial and Parallel Workflows Using CrewAI Agents**


**Objective:** To demonstrate how to build intelligent multi-agent workflows using CrewAI that execute tasks either **sequentially (serial)** or **independently (parallel)**. This demo highlights how task interdependence and concurrency affect execution order, and how agents collaborate through shared context.

**Prerequisites:**
- A `.env` file containing API keys (see the repository README for setup instructions)
- At least one LLM provider key: **Groq**, **Hugging Face**, or **Azure OpenAI**
- A **Tavily** API key for web search capabilities

**Tools & Libraries:** Python 3.10+, CrewAI, LangChain, LiteLLM, python-dotenv, Tavily

In [ ]:
# Install core dependencies: CrewAI framework, LangChain integrations, Tavily search client, and Pydantic for data validation
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

In [ ]:
# Install LiteLLM — a unified interface that lets CrewAI talk to multiple LLM providers (Groq, HuggingFace, Azure, etc.)
%pip install litellm

In [ ]:
# Standard library and third-party imports
import os
import requests
from dotenv import load_dotenv       # Reads key-value pairs from the .env file into environment variables
from crewai import Agent, Task, Crew  # Core CrewAI building blocks: agents, tasks, and the crew orchestrator
from crewai.llm import LLM            # Wrapper that connects CrewAI agents to any LLM provider via LiteLLM
from crewai.tools import BaseTool      # Base class for creating custom tools that agents can invoke

# Load environment variables from the .env file in the project root
load_dotenv()

In [ ]:
# Install optional Azure AI Inference extras (only needed if using Azure OpenAI as the LLM provider)
%pip install "crewai[azure-ai-inference]"

In [ ]:
# ==============================================================================
# Load API credentials from the .env file
# These are never hard-coded in the notebook to keep secrets out of version control.
# ==============================================================================
AZURE_OPENAI_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_ENDPOINT    = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]
AZURE_OPENAI_DEPLOYMENT  = os.environ["AZURE_OPENAI_DEPLOYMENT"]

TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]

# ==============================================================================
# Custom Tool: Tavily Web Search
# CrewAI agents can use "tools" to interact with external services. Here we wrap
# the Tavily search API into a CrewAI-compatible tool by subclassing BaseTool.
# The agent will call this tool whenever it needs real-time web information.
# ==============================================================================
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        """Send a search query to the Tavily API and return formatted results."""
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# ==============================================================================
# LLM Provider Selection
# This demo supports three interchangeable LLM backends. All expose an
# OpenAI-compatible chat-completions endpoint, so CrewAI can interact with them
# through LiteLLM without any code changes beyond swapping the config below.
#
# Change LLM_PROVIDER to "huggingface", "groq", or "azure" to switch providers.
# ==============================================================================
LLM_PROVIDER = "groq"

if LLM_PROVIDER == "huggingface":
    # Hugging Face Inference API — free-tier model via the OpenAI-compatible router
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct",
        provider="openai",
        api_key=os.getenv("HF_API_KEY"),
        base_url="https://router.huggingface.co/v1",
        temperature=0.5,
    )
elif LLM_PROVIDER == "groq":
    # Groq — ultra-fast inference (LPU) via the OpenAI-compatible endpoint
    llm = LLM(
        model="llama-3.3-70b-versatile",
        provider="openai",
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
        temperature=0.5,
    )
elif LLM_PROVIDER == "azure":
    # Azure OpenAI — enterprise-grade deployment via Azure's endpoint
    llm = LLM(
        model=AZURE_OPENAI_DEPLOYMENT,
        provider="azure",
        api_key=AZURE_OPENAI_API_KEY,
        base_url=AZURE_OPENAI_ENDPOINT,
        api_version=AZURE_OPENAI_API_VERSION,
        temperature=0.5,
    )
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}. Use 'huggingface', 'groq', or 'azure'.")

In [ ]:
# ==============================================================================
# Define Agents
# An Agent in CrewAI is an autonomous unit with a role, goal, backstory, and
# optional tools. The backstory guides the agent's persona and reasoning style.
# ==============================================================================

# Agent 1 — Researcher: gathers real-time information from the web using the
# Tavily search tool and synthesizes findings into structured notes.
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,            # Print step-by-step reasoning to the console
    allow_delegation=False,  # Do not hand off work to other agents
    llm=llm,
    tools=[search_tool]      # Give this agent access to the web search tool
)

# Agent 2 — Writer: takes research notes and produces polished executive
# summaries. Has no tools — relies entirely on its language generation ability.
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [ ]:
# ==============================================================================
# SERIAL (Sequential) EXECUTION
# In a serial workflow, tasks run one after another. Each subsequent task can
# receive the output of a previous task via the `context` parameter.
#
# Flow:  task_research  ──▶  task_write
#        (Researcher)        (Writer — uses research output as context)
#
# This is useful when a downstream task DEPENDS on the result of an upstream task.
# ==============================================================================

# Task 1 — Research: The researcher agent searches the web and produces notes
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements "
        "in AI for healthcare in 5-6 sentences. "
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

# Task 2 — Write: The writer agent consumes the researcher's output (`context`)
# and produces a polished executive summary.
task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by "
        "the AI Researcher. Limit the answer to about 200 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]  # ← This links the writer's input to the researcher's output
)

print("\n=== SERIAL EXECUTION ===")

# Assemble the crew with both agents and tasks, then kick off execution
crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True  # Print detailed execution logs
)

serial_result = crew_serial.kickoff()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)

In [ ]:
# ==============================================================================
# PARALLEL (Concurrent) EXECUTION
# In a parallel workflow, tasks run independently at the same time. Neither task
# waits for the other because there is NO `context` dependency between them.
#
# Flow:  task_parallel_1  ──▶ ┐
#        (Researcher)         ├──▶  combined result
#        task_parallel_2  ──▶ ┘
#        (Writer)
#
# Key difference from serial:
#   • `async_execution=True` — tells CrewAI to start this task without blocking
#   • No `context` parameter       — tasks do not share intermediate results
# ==============================================================================

# Task 1 (async) — The researcher searches for AI drug-discovery companies
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,  # ← Run concurrently; do not block the next task
    agent=researcher
    # No `context` — this task is independent and does not depend on any other task's output
)

# Task 2 — The writer independently produces a report on AI in diagnostics
task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 200 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

# Assemble the crew — CrewAI will schedule async tasks concurrently
crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = crew_parallel.kickoff()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)